In [ ]:
# Purpose - assess RH conditions at different y cuts (with correct normalizations)

findex = 250 #232 #-73 #163 #143 #113 # 64  63 #120

y = 6000
mysteps = list(range(338, 348))

In [ ]:
# Imports and Folder Selection

%load_ext autoreload
%autoreload 2

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun
from FieldLineHelper import FieldLineHelper 

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

my_steps = mysteps
nsteps = len(my_steps)


print(my_steps)

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields", includes_pressure = True)


In [ ]:
# Choose Regions of Interest

xlim = run.xlim
ylim = run.ylim
rect = [0, run.xlim, 0, run.ylim]

xstrips = [None] * nsteps

xcut_y = y

for ts in range(nsteps):

    roi = fq_cut(d.fields, ts, rect)

    #xcut_y = ylim / 4 * 0.75
    ycut_x = xlim / 2

    xstrips[ts] = fq_xcut(roi, xcut_y)

In [ ]:
strip = sum(xstrips) / len(xstrips)

In [ ]:
plt.figure()
plt.plot(strip.x, strip.pb)
plt.title("Magnetic Pressure (pb)")
plt.xlabel("x")
plt.ylabel("pb")
plt.show()

crossing_index = np.where(np.diff(strip.pb > 0.5))[0][1]
crossing_x = strip.x[crossing_index]
print(f"pb first crosses 0.5 at x = {crossing_x.values}")

x_position_of_min_pb = strip.x.sel(x=strip.pb.idxmin(dim="x").compute())
print(f"pb minimum occurs at x = {x_position_of_min_pb.values}")

In [ ]:
# Shock Normal Angle (theta) - compute via several independent methods

# x-positions representing (roughly) uniform upstream / downstream plasma, away from the
# shock transition layer. Tune these using the pb/dens plots above.
upstream_pos, downstream_pos = 520, 425

transition = strip.sel(x=slice(min(upstream_pos, downstream_pos), max(upstream_pos, downstream_pos)))


def _state(pos):
    return dict(
        B=np.array([strip.bx.sel(x=pos, method="nearest").values,
                    strip.by.sel(x=pos, method="nearest").values,
                    strip.bz.sel(x=pos, method="nearest").values]),
        V=np.array([strip.velx.sel(x=pos, method="nearest").values,
                    strip.vely.sel(x=pos, method="nearest").values,
                    strip.velz.sel(x=pos, method="nearest").values]),
        rho=float(strip.dens.sel(x=pos, method="nearest").values),
    )


up, down = _state(upstream_pos), _state(downstream_pos)


def orient_theta(theta):
    # shock normal should point from downstream -> upstream, i.e. nx > 0
    if np.cos(theta) < 0:
        theta += np.pi
    return (theta + np.pi) % (2 * np.pi) - np.pi


def theta_from_vec(nvec):
    nvec = np.asarray(nvec, dtype=float)[:2]
    return orient_theta(np.arctan2(nvec[1], nvec[0]))


def mass_flux_theta(rho_u, Vu, rho_d, Vd):
    # continuity: rho_u (Vu . n) = rho_d (Vd . n)  ->  A cos(theta) + B sin(theta) = 0
    A = rho_u * Vu[0] - rho_d * Vd[0]
    B = rho_u * Vu[1] - rho_d * Vd[1]
    return orient_theta(np.arctan2(-A, B))


def coplanarity_theta(Bu, Bd):
    # magnetic coplanarity theorem: n || (Bu x Bd) x (Bu - Bd)
    n = np.cross(np.cross(Bu, Bd), Bu - Bd)
    return theta_from_vec(n)


def minimum_variance_theta(transition):
    B = np.stack([transition.bx.values, transition.by.values, transition.bz.values], axis=-1)
    B = B - B.mean(axis=0)
    M = np.einsum('ij,ik->jk', B, B) / len(B)
    eigvals, eigvecs = np.linalg.eigh(M)
    n = eigvecs[:, np.argmin(eigvals)]  # minimum-variance direction
    return theta_from_vec(n)


def de_hoffmann_teller_theta(transition, rho_u, Vu, rho_d, Vd):
    # least-squares fit of E_i = -V_HT x B_i over the transition (Khrabrov & Sonnerup)
    Bv = np.stack([transition.bx.values, transition.by.values, transition.bz.values], axis=-1)
    Ev = np.stack([transition.ex.values, transition.ey.values, transition.ez.values], axis=-1)
    Bsq = np.sum(Bv ** 2, axis=-1)
    K = (Bsq[:, None, None] * np.eye(3) - Bv[:, :, None] * Bv[:, None, :]).mean(axis=0)
    V_HT = np.linalg.solve(K, np.cross(Ev, Bv).mean(axis=0))
    # boost into the HT frame, then re-apply mass-flux continuity to pin down the normal
    theta = mass_flux_theta(rho_u, Vu - V_HT, rho_d, Vd - V_HT)
    return theta, V_HT


thetas = {}
thetas['mass_conservation'] = mass_flux_theta(up['rho'], up['V'], down['rho'], down['V'])
thetas['coplanarity'] = coplanarity_theta(up['B'], down['B'])
thetas['minimum_variance'] = minimum_variance_theta(transition)
thetas['de_hoffmann_teller'], V_HT = de_hoffmann_teller_theta(transition, up['rho'], up['V'], down['rho'], down['V'])

print(f"de Hoffmann-Teller frame velocity: V_HT = ({V_HT[0]:.4f}, {V_HT[1]:.4f}, {V_HT[2]:.4f}) c\n")
for name, th in thetas.items():
    print(f"{name:>20s}: theta = {th * 180 / np.pi:7.2f} deg")

In [ ]:
# Choose which theta estimate to use

theta_method = 'mass_conservation'  # 'mass_conservation' | 'coplanarity' | 'minimum_variance' | 'de_hoffmann_teller'
manual_theta_deg = None  # set a number (degrees) to override thetas[...] entirely

if manual_theta_deg is not None:
    theta = manual_theta_deg * np.pi / 180
    print(f"Theta set manually: {manual_theta_deg:.2f} deg")
else:
    theta = thetas[theta_method]
    print(f"Theta set via '{theta_method}': {theta * 180 / np.pi:.2f} deg")

nx, ny, nz = np.cos(theta), np.sin(theta), 0.0
tx, ty, tz = np.sin(theta), -np.cos(theta), 0.0
angle_in_deg = theta * 180 / np.pi

print(f"Shock normal vector: n = ({nx:.3f}, {ny:.3f}, {nz:.3f})")
print(f"Shock tangent vector: t = ({tx:.3f}, {ty:.3f}, {tz:.3f})")

plt.figure()
plt.quiver(0, 0, nx, ny, color='r', scale=1, scale_units='xy', angles='xy', label='n')
plt.quiver(0, 0, tx, ty, color='b', scale=1, scale_units='xy', angles='xy', label='t')
plt.xlim(-1, 1); plt.ylim(-1, 1)
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.gca().set_aspect('equal', adjustable='box')
plt.title(f"Shock Normal and Tangent Vectors (angle = {angle_in_deg:.2f} deg)")
plt.legend()
plt.show()

In [ ]:
# BIG PLOT SETUP

X_CUTOFF = 0.95


def plot_squant(axis, quantity):
    right_val = strip[quantity].sel(x=X_CUTOFF * run.xlim, method="nearest")
    axis.plot(strip.x, strip[quantity] / right_val)


strip['v_n'] = strip.velx * nx + strip.vely * ny + strip.velz * nz # in units of c
strip['v_t'] = strip.velx * tx + strip.vely * ty + strip.velz * tz
strip['B_n'] = (strip.bx * nx + strip.by * ny + strip.bz * nz)
strip['B_t'] = (strip.bx * tx + strip.by * ty + strip.bz * tz)  

strip['pn'] = strip.px * nx ** 2 + strip.py * ny ** 2

strip['v_dot_B'] = strip.bx * strip.velx + strip.by * strip.vely + strip.bz * strip.velz
strip['v_sqrd'] = strip.velx ** 2 + strip.vely ** 2 + strip.velz ** 2

PNORM = 2 / run.sigma # should go in front of density * v^2 to get units compatible with pressure and magnetic field

PTI = 5.0 / 3.0 # polytropic index

In [ ]:
# Single plot of the Poynting flux term (normalized to right-side value)
fig, ax = plt.subplots(figsize=(8, 4))

boundary = strip.x[np.where(np.diff(strip.pb > 0.5))[0][1]]

strip['mass_term'] = PNORM * (strip.ndens1 + strip.ndens2 * run.mime) * strip.v_n * strip.v_t # TODO hacky before PNORM
strip['em_term'] = - 2 * strip.B_n * strip.B_t

strip['poynting_test'] = strip['mass_term'] + strip['em_term']

right_val = strip['poynting_test'].sel(x=X_CUTOFF * run.xlim, method="nearest")
ax.plot(strip.x, strip['poynting_test'] / right_val, label='poynting / right')
ax.plot(strip.x, strip['mass_term'] / right_val, label='mass term / right')
ax.plot(strip.x, strip['em_term'] / right_val, label='EM term / right')
ax.axvline(float(boundary), color='green', linestyle='--')
ax.set_xlim(x_position_of_min_pb, run.xlim * X_CUTOFF)
ax.set_xlabel("$x$ [$c / \\omega_p$]")
ax.set_ylabel("poynting / poynting_right")
ax.set_title("Poynting Flux (theta = {:.2f} deg)".format(angle_in_deg))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# RH Conditions

fig, axes = plt.subplots(3, 2, figsize=(8, 4), sharex=True)
ax1, ax2, ax3, ax4, ax5, ax6 = axes[0, 0], axes[1, 0], axes[2, 0], axes[0, 1], axes[1, 1], axes[2, 1]

plt.subplots_adjust(hspace=0, wspace = 0.2)


eqn_fsize = 18
AXIS_FONTSIZE = plt.rcParams['font.size'] * 0.7  # tick/axis-label size for this plot, 30% smaller than default

# Mass

strip['mass_flux'] = strip.dens * strip.v_n #TODO fix 

plot_squant(ax1, 'mass_flux')
#ax1.set_title(r"$\rho v_n$")
ax1.text(0.45, 0.10, r"$\rho v_n$", transform=ax1.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax1.set_ylim(0, 2.2)

# Pressure

ram = PNORM * strip.dens * strip.v_n ** 2
strip['pressure'] = ram + strip.pn + strip.pb #changed from px


# TODO fix px (think this has been fixed)

plot_squant(ax2, 'pressure')
#ax2.set_title(r"$\rho v_n^2 + P_g + P_B$")
ax2.set_ylim(0, 1.4)
ax2.text(0.35, 0.10, r"$\rho v_n^2 + P_g + P_B$", transform=ax2.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')


# Energy

strip['energy'] = strip.v_n * (0.5 * PNORM * strip.dens * strip.v_sqrd + \
                               PTI / (PTI - 1) * strip.pn + 2 * strip.pb) - \
                                2 * strip.B_n * strip.v_dot_B

plot_squant(ax3, 'energy')
#ax3.set_title("Energy")

energy_expression = r'$\rho v_n \left(\frac{v^2}{2} + \frac{\gamma}{\gamma - 1} \frac{P_g}{\rho} + \frac{B^2}{4 \pi \rho}\right) - \frac{B_n (\mathbf{v} \cdot \mathbf{B})}{4 \pi}$'
ax3.text(0.15, 0.10, energy_expression, transform=ax3.transAxes, fontsize=12, verticalalignment='bottom')
ax3.set_ylim(0, 4.2)

# B_N

plot_squant(ax4, 'B_n')
#ax4.set_title("$B_n$")
ax4.text(0.45, 0.10, '$B_n$', transform=ax4.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax4.set_ylim(0, 2.2)

# Poynting Flux

strip['poynting'] = PNORM * strip.dens * strip.v_n * strip.v_t - 2 * strip.B_n * strip.B_t #TODO fix
plot_squant(ax5, 'poynting')
#ax5.set_title(r"$\rho v_n v_t - B_n B_t / 4 \pi$")
ax5.text(0.35, 0.10, r'$\rho v_n v_t - \frac{B_n B_t} {4 \pi}$', transform=ax5.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax5.set_ylim(0, 2.2)




# Electric Field

strip['E_ideal'] = -strip.v_n * strip.B_t + strip.v_t * strip.B_n # I flipped this so it is Ez_ideal

plot_squant(ax6, 'E_ideal')
ax6.text(0.35, 0.10, '$B_n v_t - v_n B_t$', transform=ax6.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax6.set_ylim(0, 2.2)

#ax6.set_title("$B_n v_t - v_n B_t$")



for ax_row in axes:
    for ax in ax_row:
        #ax.legend()
        ax.set_xlim(x_position_of_min_pb, run.xlim * X_CUTOFF)
        #ax.set_ylim(0, 2.4)
        boundary = strip.x[np.where(np.diff(strip.pb > 0.5))[0][1]]
        ax.axvline(boundary, color = 'green', linestyle = '--')
        ax.tick_params(axis='both', which='major', labelsize=AXIS_FONTSIZE)

  

for ax in [ax3, ax6]:
    ax.set_xlabel("$x$ [$c / \omega_p$]", fontsize=AXIS_FONTSIZE)

#plt.suptitle("Rankine-Hugoniot Conditions \n" + run.boilerplate + f"\n steps = {my_steps}, y = {xcut_y} ({xcut_y / run.ylim:.02f})")
#plt.suptitle("Rankine-Hugoniot Conditions")

plt.suptitle(f"y = {xcut_y} comp, theta = {angle_in_deg:.2f} deg \n method = {theta_method} \n times = {my_steps}", fontsize=10)
plt.show()

#save_polished_fig("rh")

print(strip.x[np.isclose(strip.pb, 0.5, atol=1e-2)])

In [ ]:
# Save RH Plot Data
#
# Packages everything needed to recreate the "rh_mime" and "shock_cut_mime" figures
# (see notebooks/polished/rh_conds_polished.ipynb) into a single NetCDF file living
# alongside this notebook.

rh_save_vars = ['mass_flux', 'pressure', 'energy', 'B_n', 'poynting', 'E_ideal', 'pb', 'ndens', 'pn', 'jz']
rh_data = strip[rh_save_vars].copy()

# pn is a bare scalar NaN when pressure data isn't available (px/py have no x-dependence
# in that case) - broadcast it across x so it round-trips through NetCDF and plots cleanly.
rh_data['pn'] = rh_data['pn'] + 0 * rh_data['ndens']

rh_data.attrs.update({
    'findex': findex,
    'my_steps': list(my_steps),
    'xcut_y': xcut_y,
    'X_CUTOFF': X_CUTOFF,
    'xlim': float(run.xlim),
    'x_position_of_min_pb': float(x_position_of_min_pb),
    'boundary': float(boundary),
    'angle_in_deg': float(angle_in_deg),
    'theta_method': theta_method if manual_theta_deg is None else f"manual ({manual_theta_deg} deg)",
    'sigma': float(run.sigma),
    'mime': float(run.mime),
    'boilerplate': run.boilerplate,
})

rh_save_path = f"rh_conds_data_{findex}.nc"
rh_data.to_netcdf(rh_save_path)
print(f"Saved RH plot data to {rh_save_path}")

In [ ]:
var = 'velx'
lims = [None, None]

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(strip.x, strip[var], label=var)

ax.axvline(boundary, color='green', linestyle='--')
ax.set_xlim(x_position_of_min_pb, run.xlim * X_CUTOFF)
ax.set_xlabel("$x$ [$c / \\omega_p$]")
ax.set_ylabel(f"{var} / {var}_{{right}}")
ax.set_title(var)
ax.legend()
ax.set_ylim(*lims)

plt.tight_layout()
plt.show()

In [ ]:
energy_expression = r'\rho v_n \left(\frac{v^2}{2} + \frac{\gamma}{\gamma - 1} \frac{P_g}{\rho} + \frac{B^2}{4 \pi \rho}\right) - \frac{B_n (\mathbf{v} \cdot \mathbf{B})}{4 \pi}'
energy_expression = r'$\rho v$'

plt.figure()
plt.title(energy_expression)
plt.show()

In [ ]:
plt.rcParams.update({'font.size': 18})

#max_jz = strip.jz.sel(x = slice(len(strip.jz) / 2, len(strip.jz))).max() #NOTE different def
max_jz = strip.jz.max()

plt.figure()
#plt.axvline(boundary, color = 'green', linestyle = '--')
#plt.plot(strip.x, np.log(strip.dens), label = 'log n')
plt.plot(strip.x, strip.dens, label = 'n')
plt.plot(strip.x, -strip.B_t, label = 'B_t')

plt.xlim(run.xlim / 2, run.xlim * 0.95)
plt.xlabel("$x$ [$c / \omega_p$]")
plt.legend()
plt.show()
plt.tight_layout()

plt.rcParams.update({'font.size': 12})

#save_polished_fig("shock_cut")

In [ ]:
plt.rcParams.update({'font.size': 18})

#max_jz = strip.jz.sel(x = slice(len(strip.jz) / 2, len(strip.jz))).max() #NOTE different def
max_jz = strip.jz.max()

plt.figure()
plt.axvline(boundary, color = 'green', linestyle = '--')
plt.plot(strip.x, strip.dens, label = "$n / n_0$")
plt.plot(strip.x, strip.pn, label = "$P_g / P_{B0}$")
plt.plot(strip.x, strip.pb, label = "$P_B / P_{B0}$")
plt.plot(strip.x, strip.jz / max_jz, label = "$j_z / j_{\mathrm{max}}$") #/ max_jz
plt.plot(strip.x, strip.jz * strip.ez / 1e-2, label = "$J_z E_z$") 


plt.xlim(run.xlim / 2, run.xlim * 0.95)
plt.xlabel("$x$ [$c / \omega_p$]")
plt.legend()
plt.show()
plt.tight_layout()

plt.rcParams.update({'font.size': 12})

#save_polished_fig("shock_cut")

In [ ]:
# Slow-Mode Shock Conditions



fig, axes = plt.subplots(3, 2, figsize=(8, 6))
ax1, ax2, ax3, ax4, ax5, ax6 = axes[0, 0], axes[1, 0], axes[2, 0], axes[0, 1], axes[1, 1], axes[2, 1]

def degree_formatter(x, pos):
    return f"{x:.0f}°"

strip['B_mag'] = strip.bx ** 2 + strip.by ** 2 + strip.bz ** 2
strip['theta_B'] = np.arccos(strip.B_n / strip.B_mag)

strip['v_A'] = np.sqrt(run.sigma) * strip.B_mag
strip['v_I'] = strip.v_A * np.cos(strip.theta_B)
strip['v_S'] = np.sqrt(run.sigma / 2 * PTI * strip.px / strip.dens)
strip['v_SM'] = np.sqrt(0.5) * np.sqrt(strip.v_S ** 2 + strip.v_A ** 2 - \
                                       np.sqrt((strip.v_S ** 2 + strip.v_A ** 2) ** 2 - 4 * strip.v_S ** 2 * strip.v_I ** 2))

strip['M_I'] = strip.v_n / strip.v_I 
strip['M_SM'] = strip.v_n / strip.v_SM

# Plasma Pressure

plot_squant(ax1, 'px')
ax1.set_title(r"$P_x$ (increase)")

# Magnetic Pressure

plot_squant(ax2, 'pb')
ax2.set_title(r"$P_B$ (decrease)")
ax2.set_ylim(0)

# Angle
strip['theta_B_deg'] = 180 / np.pi * np.arccos(strip.B_n / strip.B_mag)

plot_squant(ax3, 'theta_B_deg')
ax3.set_title(r"$\theta_B$ (decrease)")
ax3.yaxis.set_major_formatter(FuncFormatter(degree_formatter))

# M_I

plot_squant(ax4, 'M_I')
ax4.set_title(r"$M_I$ ($\leq 1$ upstream)")

# M_SM

plot_squant(ax5, 'M_SM')
ax5.set_title(r"$M_{SM}$ ($> 1 \; \rightarrow \; < 1$)")

for ax in [ax4, ax5]:
    ax.axhline(1, color = 'black', linestyle = '--')
    ax.axhline(0, color = 'grey', linestyle = ':', alpha = 1)

for ax_row in axes:
    for ax in ax_row:
        #ax.legend()
        ax.set_xlim(run.xlim / 2, run.xlim)

        for bound in strip.x[np.isclose(strip.pb, 0.5, atol=1e-2)]:
            ax.axvline(bound, color = 'green', linestyle = '--')



plt.suptitle("Slow-Mode Shock Conditions \n" + run.boilerplate + f"\n steps = {my_steps}")

plt.tight_layout()
plt.show()



print(strip.x[np.isclose(strip.pb, 0.5, atol=1e-2)])